# Laboratorio 14 — Por dentro de una IA de lenguaje
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701
**Fecha:** Miércoles 25 de noviembre de 2026 · Clase 14 · Lab. Central 114
**Horario del lab:** ~11:50–13:35 (105 min)
**Viene de:** deck «Clase 14 - Transformers y LLM»: diapo 3 (un LLM juega a adivinar lo que sigue), diapos 7–8 (todos miran a todos; varias cabezas), diapo 12 (del texto a los números: pedacitos), diapo 13 (las fichas forman un mapa de significados), diapo 15 (temperatura), diapo 16 (de loro que completa a asistente que responde), diapo 18 (alucinar), diapo 19 (buscar por significado), diapo 21 (fotos y textos en un mismo mapa), diapo 22 (un LLM que mira la foto) y diapo 23 (de ruido a dibujo).

| Parte | Tema | Tiempo sugerido |
|---|---|---|
| **0** | Apertura y configuración (mientras descarga) | 5 min |
| **1** | Pedacitos (con el tokenizador real) | 12 min |
| **2** | El autocompletar gigante (lo escribes tú) | 30 min |
| **3** | El mapa de significados, tu buscador y RAG a mano | 28 min |
| **4** | Imágenes: clasificar, describir y dibujar | 18 min |
| **Cierre** | Entregable | 10 min |
| **Extras** | E1 la atención real · E2 la atención a mano · E3 tu proyecto | opcional |

## Objetivos de aprendizaje

1. Ver cómo «lee» el texto una IA: no ve letras, ve **pedacitos** con un número cada uno, y usar el tokenizador real.
2. Ver que un LLM es un **autocompletar gigante**, **escribiendo tú ese autocompletar** con el modelo de verdad; mover **la perilla de la creatividad** y ver qué recibe por dentro **el asistente**.
3. Ver frases convertidas en **puntos de un mapa de significados**, **programar tu propio buscador** y armar a mano el mensaje con el documento correcto, para comprobar que así el asistente inventa menos.
4. Usar IAs que clasifican fotos con frases escritas por ti y que describen una foto, y ver cómo se dibuja una imagen desde ruido.

> **Qué necesitas.** Para este lab te recomendamos **Google Colab** (`Archivo → Subir notebook`; basta la CPU): los modelos pesan unos 4 GB en total y en Colab se descargan por la red de Google, no por el wifi de la sala. También corre en tu computador con Python 3.13 y `pip install -r requirements.txt`. Todo funciona en la CPU. Si la descarga falla, el notebook entra en **modo sin red** y muestra las salidas que grabó el profe: las preguntas se pueden responder igual.

> **Cómo se trabaja.** Completa las celdas marcadas con `# TODO` (reemplaza los `...`). Justo después de cada una hay un **autochequeo**: una celda que te avisa con un mensaje si algo no cuadra. Si pasa, verás un ✔. Las preguntas se responden en la celda «**Respuesta:**» que sigue a cada una. Los dibujos vienen listos en `llm_utils.py`, pero las piezas importantes las escribes tú con los modelos de verdad, que abres con `abrir_modelo(...)`. Para ver el código real de cualquier herramienta, usa `ver_codigo(nombre)`. Si un error menciona `ellipsis`, quedó un `...` sin reemplazar.

In [ ]:
# --- Configuración -----------------------------------------------------------
import sys, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
URL_REPO = "https://raw.githubusercontent.com/diegoalrv/ml-ein143a25/main/14/"
if IN_COLAB:
    %pip install -q "transformers>=4.56" sentence-transformers scikit-image
if not Path("llm_utils.py").exists():  # las herramientas compartidas del lab (vienen junto al notebook)
    import urllib.request
    urllib.request.urlretrieve(URL_REPO + "llm_utils.py", "llm_utils.py")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import llm_utils
from llm_utils import (AZUL, GRIS, MARINO, NARANJA, DISTRACTORES, PREGUNTAS_CURSO, barras_siguiente, buscar,
                       clasificar_foto, comparar_etapas, completar, describir_foto, fichas, fotos_ejemplo, lado_a_lado,
                       mapa_atencion, mapa_significados, mostrar_galeria, mostrar_pedacitos, parecido,
                       perilla_temperatura, porcentajes_siguiente, precargar, preguntar, subir_foto,
                       abrir_modelo, ver_codigo)

SEED = 42
RAPIDO = False   # True = textos generados más cortos, solo para ensayar el código
llm_utils.RAPIDO = RAPIDO
np.random.seed(SEED)
T0 = time.time()

# Paleta del deck «Clase 14 - Transformers y LLM»
plt.rcParams.update({
    "figure.facecolor": "#f6f6f4", "savefig.facecolor": "#f6f6f4", "axes.facecolor": "#fffefb",
    "axes.edgecolor": "#dcdad3", "axes.labelcolor": MARINO, "text.color": MARINO,
    "xtick.color": "#5d6675", "ytick.color": "#5d6675", "axes.grid": True, "grid.color": "#e3e1da",
    "grid.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlecolor": MARINO, "figure.dpi": 100, "font.size": 10.5,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, MARINO, GRIS]),
})
import torch, transformers
print("✔ Listo · Python", sys.version.split()[0], "| PyTorch", torch.__version__, "| transformers",
      transformers.__version__, "| RAPIDO =", RAPIDO, "| Colab =", IN_COLAB)
if not IN_COLAB:
    print("  (Para este lab conviene Colab: las descargas no pasan por el wifi de la sala.)")

---
## Parte 0 — Apertura (5 min)

> **En palabras simples.** Hoy usamos IAs de verdad, chiquititas, que caben en un computador normal. Son primas pequeñas de ChatGPT, Claude o Gemini: misma receta, mucho menos tamaño. Las vamos a abrir en cuatro paradas: cómo ven el texto, cómo eligen qué decir, cómo entienden el significado y cómo miran fotos.

Estos son los modelos de hoy. Ejecuta la celda de abajo **ahora**: carga los dos primeros mientras el profe introduce el lab (la primera vez demora 1 o 2 minutos).

**No necesitas cuenta de Hugging Face** (el sitio de donde se bajan los modelos): todos los de hoy son públicos. Si Colab avisa que falta `HF_TOKEN`, ignora el aviso. Si una celda se queda pegada más de 3 minutos cargando un modelo, detenla (botón ■) y vuelve a ejecutarla.

In [ ]:
display(pd.DataFrame(
    [(quien, tam, repos[0]) for repos, quien, tam in llm_utils.MODELOS.values()],
    columns=["cómo le diremos", "descarga", "nombre en Hugging Face"],
    index=pd.Index(["Partes 1 y 2", "Parte 2", "Parte 3", "Parte 4.1", "Parte 4.2"], name="se usa en")))
precargar("base", "fichas")   # el que solo completa y el de fichas; los otros se bajan cuando se usen

---
## Parte 1 — Pedacitos (12 min)

> **En palabras simples (diapo 12).** La IA no lee letras ni palabras: corta el texto en **pedacitos** frecuentes y a cada pedacito le da un número. En los libros, a cada pedacito se le llama ***token***; aquí les diremos pedacitos.

`mostrar_pedacitos(texto)` dibuja cada pedacito en una cajita de color, con su número debajo. Los espacios se ven como `·`: el espacio va pegado al pedacito que sigue.

In [ ]:
mostrar_pedacitos("Machine learning es genial")
mostrar_pedacitos("Machine learning is great")

### 1.1 El tokenizador de verdad

`mostrar_pedacitos` es solo un dibujo. Por dentro usa el **tokenizador** del modelo: la pieza real que corta el texto. Lo abrimos con `abrir_modelo("base")`, que entrega el tokenizador (`tok`) y el modelo de verdad (`modelo_base`), los mismos que usa ChatGPT en versión gigante. Con `tok.encode(texto)` el texto pasa a números y con `tok.decode(numeros)` los números vuelven a texto.

In [ ]:
tok, modelo_base = abrir_modelo("base")   # el tokenizador y el modelo de verdad (Qwen2.5-0.5B)

numeros = tok.encode("Machine learning es genial")
print("Números:  ", numeros)
print("Pedacitos:", [tok.decode([n]) for n in numeros])
print("De vuelta:", tok.decode(numeros))
print(f"Este tokenizador conoce {len(tok):,} pedacitos distintos.".replace(",", "."))

### TODO 1 — Tus frases en pedacitos

Escribe **3 frases tuyas**. Al menos una con chilenismos (ideas: «cachai», «po», «Talcahuano», «sopaipilla», «al tiro») y, si quieres, una en inglés para comparar. Después arma la lista `cuantos`, con cuántos pedacitos tiene cada frase según el tokenizador (pista: `len(tok.encode(frase))`, recorriendo `mis_frases`).

In [ ]:
# TODO 1: 3 frases tuyas (al menos una con chilenismos) y cuántos pedacitos tiene cada una
mis_frases = [
    ...,
    ...,
    ...,
]
cuantos = ...   # una lista: para cada frase de mis_frases, len(tok.encode(frase))

In [ ]:
# --- Autochequeo: tus frases ---
assert isinstance(mis_frases, list) and len(mis_frases) == 3, "mis_frases debería ser una lista con exactamente 3 frases."
assert all(isinstance(f, str) and f.strip() for f in mis_frases), \
    "Cada frase va entre comillas y no puede quedar vacía: reemplaza los ... por tus frases."
assert cuantos == [len(tok.encode(f)) for f in mis_frases], \
    "cuantos debería tener un número por frase: len(tok.encode(frase)). Pista: [... for frase in mis_frases]"
print("✔ Tres frases listas")
for frase, n in zip(mis_frases, cuantos):
    print(f"  {n:3d} pedacitos · {len(frase):3d} letras · {frase}")

In [ ]:
for frase in mis_frases:
    mostrar_pedacitos(frase)

### Mini-desafío: ¿cuántas letras «r» tiene «frutilla»?

Para ti es fácil: una. Mira lo que ve la IA.

In [ ]:
mostrar_pedacitos("frutilla")

La IA no ve «f-r-u-t-i-l-l-a»: ve 3 pedacitos convertidos en números. Para contar las «r» tendría que acordarse de cómo se deletrea cada pedacito, algo que nunca mira directamente. Por eso a los chatbots les cuesta contar letras (el ejemplo famoso es «¿cuántas r tiene *strawberry*?»).

**Pregunta 1.1.** ¿Qué frase usó más pedacitos, la de español o la de inglés? ¿Qué pasa con las palabras chilenas? ¿Por qué importa eso si cada pedacito cuesta tiempo y plata?

**Respuesta:** (escribe aquí)

---
## Parte 2 — El autocompletar gigante (30 min)

### 2.1 Los porcentajes (8 min)

> **En palabras simples (diapo 3).** La IA mira todo lo escrito y le pone un **porcentaje** a cada pedacito posible para lo que viene. Después elige uno, lo pega al final y repite. Así, pedacito a pedacito, escribe todo.

`barras_siguiente(texto)` muestra los 10 pedacitos con más porcentaje según **el que solo completa** (el modelo recién entrenado con texto de internet).

In [ ]:
barras_siguiente("La capital de Chile es")
barras_siguiente("La capital de Chile es la ciudad de")
barras_siguiente("La mejor ciudad de Chile es")

**Fíjate.** Con «La capital de Chile es» el modelo **no** salta directo a «Santiago»: lo más probable es « una» o « la» (como en «es una ciudad…»). Recién con «…es la ciudad de», «Santiago» se lleva la mayoría. El modelo no piensa la respuesta entera: solo adivina el pedacito siguiente. Y en la tercera, la de la diapo 3, entre los candidatos aparece « Buenos» (de Buenos Aires, que ni siquiera está en Chile).

**Así se calculan, a mano.** Le pasamos los números del texto a `modelo_base` y nos devuelve un **puntaje** para cada pedacito posible (más de 150.000). `torch.softmax` convierte esos puntajes en porcentajes que suman 100 (es lo mismo que usan las redes que clasifican, como la CNN de la Clase 11) y `torch.topk` se queda con los 5 mayores.

In [ ]:
numeros = tok.encode("La capital de Chile es la ciudad de")
with torch.no_grad():                                                  # solo usamos el modelo, no lo entrenamos
    puntajes = modelo_base(torch.tensor([numeros])).logits[0, -1]     # un puntaje por cada pedacito posible
porcentajes = torch.softmax(puntajes, dim=0) * 100                    # de puntajes a porcentajes que suman 100
mejores = torch.topk(porcentajes, 5)
print("Un puntaje por cada pedacito posible. Los 5 con más porcentaje:")
for p, n in zip(mejores.values, mejores.indices):
    print(f"  {tok.decode([int(n)])!r:>12}  {float(p):5.1f} %")

### TODO 2 — Tu propio comienzo

Escribe el **comienzo** de una frase, sin punto final (para que quede algo por adivinar), y mira sus porcentajes. Pista: si terminas en «…es el» o «…es la», la IA tiene que adivinar una palabra con contenido y no solo un «el» o un «que».

In [ ]:
# TODO 2: el comienzo de una frase, sin punto final
mi_comienzo = ...

In [ ]:
# --- Autochequeo: tu comienzo ---
assert isinstance(mi_comienzo, str) and len(mi_comienzo.strip()) > 3, "Escribe el comienzo de una frase entre comillas."
assert not mi_comienzo.strip().endswith((".", "!", "?")), \
    "Sácale el punto final: la IA tiene que adivinar cómo sigue la frase."
mi_comienzo = mi_comienzo.rstrip()   # un espacio al final confunde a la IA
print("✔ Comienzo listo")
barras_siguiente(mi_comienzo)

### 2.2 Escribe tú el autocompletar (7 min)

Escribir un texto completo es solo repetir lo anterior: pedir el pedacito más probable, pegarlo al final y volver a preguntar. La función de abajo hace **una** vuelta: recibe los números del texto y devuelve el número del pedacito con más porcentaje (`argmax` = la posición del mayor, como en la CNN de la Clase 11).

In [ ]:
def numero_mas_probable(numeros):
    """El número del pedacito con más porcentaje para seguir después de `numeros`."""
    with torch.no_grad():
        puntajes = modelo_base(torch.tensor([numeros])).logits[0, -1]
    return int(puntajes.argmax())


n = numero_mas_probable(tok.encode("La capital de Chile es la ciudad de"))
print(f"Número {n}, que es el pedacito {tok.decode([n])!r}")

### TODO 3 — Tu autocompletar

Completa el `for`: en cada vuelta, **agrega al final de `numeros`** el número más probable. Con eso tu código escribe 10 pedacitos seguidos a partir de `mi_comienzo`. Es todo lo que hace un LLM para escribir (el de ChatGPT es mucho más grande, pero el ciclo es este mismo).

In [ ]:
# TODO 3: tu autocompletar (10 pedacitos, siempre el más probable)
numeros = tok.encode(mi_comienzo)
for paso in range(10):
    numeros = ...   # numeros + [el número más probable para seguir]
mi_texto = tok.decode(numeros)
print(mi_texto)

In [ ]:
# --- Autochequeo: tu autocompletar ---
assert isinstance(numeros, list), "numeros debería seguir siendo una lista: numeros + [ ... ]"
nuevos = len(numeros) - len(tok.encode(mi_comienzo))
assert nuevos == 10, f"Deberían ser 10 pedacitos nuevos y hay {nuevos}: agrega uno (y solo uno) por vuelta."
assert mi_texto.startswith(mi_comienzo), "mi_texto debería empezar con tu comienzo: ¿partiste de tok.encode(mi_comienzo)?"
referencia = completar(mi_comienzo, n_pedacitos=10, temperatura=0)
if mi_texto == referencia:
    print("✔ Tu autocompletar escribe exactamente lo mismo que la herramienta completar(..., temperatura=0)")
else:
    print("⚠ Casi: tu texto y el de completar(...) se separan en algún pedacito. Si en el for usaste "
          "numero_mas_probable(numeros), puede ser un empate de porcentajes; si no, revísalo.")
lado_a_lado(("Tu autocompletar", mi_texto, AZUL), ("completar(..., temperatura=0)", referencia, GRIS))

### 2.3 La perilla de la creatividad (5 min)

> **En palabras simples (diapo 15).** La **temperatura** es la perilla de la creatividad. Baja: la IA casi siempre elige el pedacito con más porcentaje (va a la segura). Alta: les da chance a los pedacitos raros (más creativa, pero se puede desordenar).

Tu autocompletar siempre elige el más probable: es la perilla en 0. Con la perilla más arriba, en vez de elegir el mayor se **sortea** un pedacito según sus porcentajes, y la perilla aplana o afila esos porcentajes antes del sorteo. La perilla parte en 0,7, donde la IA todavía escribe con sentido. Muévela a **0,2** y a **1,8**: arriba cambian las barras y abajo aparecen 3 continuaciones con esa perilla (cada movida demora unos segundos).

In [ ]:
perilla_temperatura("Mi mascota favorita es el")

Mira los pedacitos de arriba: « per» (de perro) y « g» (de gato) aparecen partidos. La IA va armando la palabra de a poco.

**Abre la caja.** Así está hecha la herramienta `completar` que usa la perilla. Busca `generate(...)`: hace tu mismo `for`, y la perilla entra como `temperature`. (Lo de `_grabado` es para el modo sin red; no importa aquí.)

In [ ]:
ver_codigo(completar)

**Pregunta 2.3.** ¿Qué pasó con la perilla en 0,2 y qué en 1,8? ¿Para qué tarea usarías temperatura baja y para cuál alta?

**Respuesta:** (escribe aquí)

### 2.4 Del loro al asistente (5 min)

> **En palabras simples (diapo 16).** El modelo recién entrenado solo sabe **continuar texto**, como un loro que leyó internet: es **el que solo completa**. Después se le muestran ejemplos de buenas respuestas y personas eligen la mejor entre varias (¡puntos por hacerlo bien, como el Mario de la clase pasada!). Así aprende a responder: se vuelve **el asistente**.

`comparar_etapas(pregunta)` le pasa la misma pregunta a los dos: al que solo completa como texto crudo (gris) y al asistente con su formato de conversación (azul).

In [ ]:
comparar_etapas("¿Cuál es la capital de Chile?")

In [ ]:
comparar_etapas("¿Qué es el machine learning?")

**¿Cómo sabe el asistente que le están preguntando?** No lo «sabe»: recibe un texto con **marcas especiales** que dicen quién habla, y lo continúa con el mismo autocompletar de siempre. Así se ve el texto que el asistente recibe de verdad cuando le preguntas algo:

In [ ]:
tok_asistente, modelo_asistente = abrir_modelo("asistente")
mensajes = [
    {"role": "system", "content": "Eres un asistente útil. Responde siempre en español, en pocas frases."},
    {"role": "user", "content": "¿Cuál es la capital de Chile?"},
]
print(tok_asistente.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True))

El asistente sigue escribiendo justo después de `<|im_start|>assistant`, pedacito a pedacito, y se detiene cuando escribe la marca `<|im_end|>`. La herramienta `preguntar(...)` arma exactamente este texto: si quieres, compruébalo con `ver_codigo(preguntar)`.

**Pregunta 2.4.** ¿Qué hizo el que solo completa con la pregunta? ¿Qué cambió en el asistente? ¿Qué tiene de especial el texto que recibe el asistente?

**Respuesta:** (escribe aquí)

### 2.5 Una alucinación en vivo (5 min)

> **En palabras simples (diapo 18).** La IA está entrenada para **sonar bien**, no para decir la verdad. Si no sabe algo, igual reparte porcentajes y escribe algo con total seguridad. A eso se le llama **alucinar**.

Le preguntamos al asistente dos cosas que **no puede saber**. Lo inventado va en naranja.

In [ ]:
PREGUNTA_CERTAMEN = "¿Cuándo es el Certamen 3 del curso Machine Learning de la USM Concepción y qué entra?"
PREGUNTA_EMPANADAS = "¿Cuál es el récord mundial de empanadas comidas en una hora?"
lado_a_lado(("¿Cuándo es el Certamen 3?", preguntar(PREGUNTA_CERTAMEN), NARANJA),
            ("¿Récord de comer empanadas?", preguntar(PREGUNTA_EMPANADAS), NARANJA))

**Pregunta 2.5.** ¿Respondió con seguridad? ¿Cómo podrías saber si es verdad? (Déjala abierta: la Parte 3 la resuelve.)

**Respuesta:** (escribe aquí)

---
## Parte 3 — El mapa de significados y el buscador (28 min)

### 3.1 Fichas y mapa (10 min)

> **En palabras simples (diapo 13).** Cada frase se convierte en una **ficha**: una lista de números (en los libros, un ***embedding***). Frases con significado parecido tienen fichas parecidas, así que quedan **cerca en el mapa de significados**, aunque usen palabras distintas.

Abrimos el modelo de fichas de verdad: `modelo_fichas.encode(texto)` entrega la ficha.

In [ ]:
modelo_fichas = abrir_modelo("fichas")
ficha_hola = modelo_fichas.encode("Hola")
print(f"La ficha de «Hola» es una lista de {len(ficha_hola)} números. Los primeros 5:")
print(np.round(ficha_hola[:5], 3))

`modelo_fichas.similarity(a, b)` compara dos fichas: 0 es «nada que ver» y 1 es «mismo significado» (aquí lo pasamos a %). La herramienta `parecido(a, b)` hace exactamente esto.¹

In [ ]:
completo, hot_dog, banco = modelo_fichas.encode(["Me encanta el completo italiano", "Adoro los hot dogs con palta",
                                                 "El banco me cobró una comisión"])
print(f"«Me encanta el completo italiano» vs «Adoro los hot dogs con palta»:   "
      f"{float(modelo_fichas.similarity(completo, hot_dog)) * 100:.0f} %")
print(f"«Me encanta el completo italiano» vs «El banco me cobró una comisión»: "
      f"{float(modelo_fichas.similarity(completo, banco)) * 100:.0f} %")

Las dos primeras no comparten **ni una palabra** y aun así se parecen mucho más: el modelo aprendió que «completo», «hot dog» y «palta» hablan de lo mismo.

<sub>¹ Para los curiosos: se calcula con la *similitud coseno* entre las dos fichas, que mide si apuntan para el mismo lado.</sub>

### TODO 4 — Tu mapa de significados

Completa los 4 grupos (vienen 2 frases de ejemplo en cada uno): agrega **al menos una frase por grupo** (mejor dos). Escribe frases que **hablen del tema**, como las de ejemplo: «La vaca da leche» se agrupa mejor que «Ayer vi una vaca en la carretera», que habla más del viaje que del animal. Puedes cambiar los grupos por otros temas. Para dibujar el mapa, las fichas de 384 números se achican a solo 2 con **PCA**, la misma de la Clase 4.

In [ ]:
# TODO 4: al menos 3 frases por grupo
frases_por_grupo = {
    "animales": ["La vaca da leche en el campo", "El perro es el mejor amigo del ser humano", ...],
    "ciudades de Chile": ["Antofagasta es una ciudad del norte de Chile", "Concepción es la capital de la Región del Biobío", ...],
    "comida": ["La empanada de pino lleva carne y cebolla", "La cazuela es una sopa con carne y verduras", ...],
    "fútbol": ["El delantero metió un gol de cabeza", "El arquero atajó un penal en el partido", ...],
}

In [ ]:
# --- Autochequeo: tus grupos ---
for grupo, lista in frases_por_grupo.items():
    assert all(isinstance(f, str) and f.strip() for f in lista), \
        f"En «{grupo}» quedó un ... o una frase vacía: reemplázalo por una frase entre comillas."
    assert len(lista) >= 3, f"El grupo «{grupo}» tiene {len(lista)} frases: agrega al menos {3 - len(lista)} más."
frases = [f for lista in frases_por_grupo.values() for f in lista]
grupos = [g for g, lista in frases_por_grupo.items() for _ in lista]
print(f"✔ {len(frases)} frases en {len(frases_por_grupo)} grupos")
mapa_significados(frases, grupos)

**Pregunta 3.1.** ¿Los grupos quedaron separados en el mapa? ¿Alguna frase quedó «perdida»? ¿Por qué crees?

**Respuesta:** (escribe aquí)

### 3.2 Tu propio buscador (10 min)

> **En palabras simples (diapo 19).** Para buscar por significado, la pregunta se convierte en ficha y se buscan los documentos con la ficha más cercana. No hace falta que compartan palabras: basta que hablen de lo mismo.

Nuestros documentos: las **preguntas frecuentes del curso** y algunos avisos que no tienen nada que ver.

In [ ]:
documentos = PREGUNTAS_CURSO + DISTRACTORES
display(pd.DataFrame({"documento": documentos}, index=pd.RangeIndex(1, len(documentos) + 1)))

Con la herramienta `buscar(pregunta, documentos)` se ven los 3 más parecidos:

In [ ]:
buscar("¿Qué día es la última prueba?", documentos)

La pregunta no dice «certamen» y aun así lo encontró primero: «prueba» y «evaluación» significan casi lo mismo.

### TODO 5 — Programa tu buscador

Ahora hazlo tú con el modelo de fichas. `mi_buscador` saca la ficha de la pregunta y la de cada documento, las compara y devuelve el documento más parecido. Te faltan dos líneas: las fichas de todos los documentos (`encode` también acepta una lista) y la posición del más parecido (`.argmax()`, como en el autocompletar). Después escribe **2 preguntas** sobre algo que esté en los documentos, pero **con otras palabras**.

In [ ]:
# TODO 5: tu buscador y dos preguntas tuyas
def mi_buscador(pregunta, documentos):
    ficha_pregunta = modelo_fichas.encode(pregunta)
    fichas_documentos = ...   # las fichas de todos los documentos
    parecidos = modelo_fichas.similarity(ficha_pregunta, fichas_documentos)[0]   # un parecido por documento
    mejor = ...               # la posición del más parecido (pista: .argmax())
    return documentos[mejor]


mis_busquedas = [..., ...]   # 2 preguntas sobre los documentos, con otras palabras

In [ ]:
# --- Autochequeo: tu buscador ---
assert isinstance(mis_busquedas, list) and len(mis_busquedas) == 2 \
    and all(isinstance(q, str) and q.strip() for q in mis_busquedas), \
    "Escribe 2 preguntas entre comillas en la lista mis_busquedas."
for q in ["¿Qué día es la última prueba?"] + mis_busquedas:
    assert mi_buscador(q, documentos) == buscar(q, documentos)[0], \
        f"Para «{q}» tu buscador eligió otro documento que buscar(...): revisa fichas_documentos y mejor."
print("✔ Tu buscador elige lo mismo que la herramienta buscar(...). Los 3 mejores de tus preguntas:")
for q in mis_busquedas:
    display(buscar(q, documentos))
print("El buscador siempre trae algo, aunque no tenga nada que ver (prueba con «hola»).\n"
      "Tú decides si el primero es el correcto: respóndelo en la Pregunta 3.2.")

**Pregunta 3.2.** ¿El buscador encontró lo que buscabas? ¿Hubo alguna búsqueda en que el primero no era el correcto?

**Respuesta:** (escribe aquí)

### 3.3 Buscar primero, responder después (8 min) · diapos 18 y 19

> **En palabras simples.** Si antes de responder le pasamos al asistente **el documento correcto**, ya no tiene que adivinar. Así funcionan los asistentes que responden con los documentos de una empresa: primero buscan y después responden (en los libros, ***RAG***).

No tiene nada de magia: es **pegar el documento en el mensaje**, antes de la pregunta.

### TODO 6 — Arma tú el mensaje

Con un f-string, arma `mensaje` con tres cosas: esta instrucción, el `documento` que encontró tu buscador y la `PREGUNTA_CERTAMEN` de la Parte 2.5.

> Responde solo con la información de este texto. Si la respuesta no está en el texto, di que no lo sabes.

In [ ]:
# TODO 6: el mensaje con la instrucción, el documento y la pregunta
documento = mi_buscador(PREGUNTA_CERTAMEN, documentos)
mensaje = ...   # un f-string con la instrucción, {documento} y {PREGUNTA_CERTAMEN}

In [ ]:
# --- Autochequeo: tu mensaje ---
assert isinstance(mensaje, str), "mensaje debería ser un texto: un f-string entre comillas."
assert documento in mensaje, "Falta pegar el documento: pon {documento} dentro del f-string."
assert PREGUNTA_CERTAMEN in mensaje, "Falta la pregunta: pon {PREGUNTA_CERTAMEN} dentro del f-string."
print("✔ Mensaje listo. Esto es todo lo que recibe el asistente:\n")
print(mensaje)
lado_a_lado(("Sin documento: inventa", preguntar(PREGUNTA_CERTAMEN), NARANJA),
            ("Con tu mensaje: documento + pregunta", preguntar(mensaje), AZUL))

¿Y si el buscador trae el documento equivocado? Probemos pasándole a propósito un aviso que no tiene nada que ver (la herramienta `preguntar(..., contexto=...)` arma el mismo mensaje que hiciste tú), y con la pregunta de las empanadas, que no está en ningún documento:

In [ ]:
lado_a_lado(("Certamen con un documento equivocado", preguntar(PREGUNTA_CERTAMEN, contexto=DISTRACTORES[0]), NARANJA),
            ("Empanadas, con lo mejor que encontró", preguntar(PREGUNTA_EMPANADAS, contexto=buscar(PREGUNTA_EMPANADAS, documentos)[0]), AZUL))

**Pregunta 3.3.** ¿Qué cambió en la respuesta? ¿Qué pasa si el buscador trae el documento equivocado?

**Respuesta:** (escribe aquí)

---
## Parte 4 — Imágenes: clasificar, describir y dibujar (18 min)

### 4.1 Fotos y frases en el mismo mapa (8 min)

> **En palabras simples (diapo 21).** Este modelo (se llama CLIP) aprendió con millones de fotos de internet con su descripción. Pone **fotos y frases en el mismo mapa de significados**, así que puede decir qué frase está más cerca de una foto, aunque nunca lo hayan entrenado con esas etiquetas. Tú escribes las etiquetas, en español.

In [ ]:
FOTOS = fotos_ejemplo()   # vienen con scikit-image: no necesitan internet
clasificar_foto(FOTOS["gato"], ["un gato", "un perro", "una taza de café", "un cohete"])

### TODO 7 — Tu foto, tus etiquetas

Sube una foto tuya con `subir_foto()`. Ideas: algo del campus, de tu mesa o tu mascota. **Sin personas.** Después escribe entre 3 y 8 etiquetas en español: una correcta y otras que se le parezcan, para ponerle difícil. Si no puedes subir una foto, usa `foto_propia = FOTOS["cohete"]`.

In [ ]:
# TODO 7a: tu foto (sin personas). Si falla la subida, sigue con el cohete.
foto_propia = subir_foto()      # si no puedes subir una foto: foto_propia = FOTOS["cohete"]

In [ ]:
# TODO 7b: entre 3 y 8 etiquetas: una correcta y otras que se le parezcan
mis_etiquetas = [..., ..., ..., ...]

In [ ]:
# --- Autochequeo: tus etiquetas ---
assert 3 <= len(mis_etiquetas) <= 8, f"Escribe entre 3 y 8 etiquetas (tienes {len(mis_etiquetas)})."
assert all(isinstance(e, str) and e.strip() for e in mis_etiquetas), "Cada etiqueta va entre comillas: reemplaza los ..."
print(f"✔ {len(mis_etiquetas)} etiquetas listas")
clasificar_foto(foto_propia, mis_etiquetas)

**Abre la caja.** Así está hecha `clasificar_foto`: saca la ficha de la foto y la de cada etiqueta con CLIP, las compara y pasa los parecidos a porcentajes con la misma `softmax` de la Parte 2.1.

In [ ]:
ver_codigo(clasificar_foto)

**Pregunta 4.1.** Compáralo con la CNN de la Clase 11, que necesitó miles de imágenes etiquetadas para aprender 10 clases fijas. ¿Qué ventaja tiene este modelo? ¿Dónde crees que se puede equivocar?

**Respuesta:** (escribe aquí)

### 4.2 Una IA que mira y describe (5 min)

> **En palabras simples (diapo 22).** La foto se corta en **parches** (cuadraditos), que funcionan como pedacitos. Después un LLM escribe una descripción pedacito a pedacito, igual que en la Parte 2.

Este modelo es muy pequeño y aprendió sobre todo en inglés, así que **responde en inglés** aunque le preguntemos en español. Se acepta: fíjate en lo que dice, no en el idioma.

In [ ]:
describir_foto(FOTOS["cafe"])

In [ ]:
describir_foto(foto_propia)

**Pregunta 4.2.** ¿Describió bien tu foto? ¿Inventó algo que no estaba? (Si inventó, es una alucinación con imágenes.)

**Respuesta:** (escribe aquí)

### 4.3 Demo del profe: de ruido a dibujo (5 min)

> **En palabras simples (diapo 23).** Para dibujar desde una frase, la IA parte de una imagen de **puro ruido** (como la tele sin señal) y en cada paso le quita un poco de ruido, guiada por la frase, hasta que aparece un dibujo. Es lo contrario de describir una foto.

El profe lo corre en vivo en su computador (`texto_a_imagen_profe.py`): **díctenle frases**. Aquí abajo queda la galería grabada, para quien no alcance a ver la proyección.

In [ ]:
mostrar_galeria()

Fíjate en «un robot tomando mote con huesillo»: el robot salió, pero el mote con huesillo no. Casi no aparece en las fotos de internet con las que aprendió, igual que lo muy local le cuesta a CLIP (Parte 4.1).

---
## Cierre — Entregable (10 min)

> **En palabras simples (diapo 26).** Todo lo de hoy es una sola cadena:
> **texto → pedacitos → fichas → [mirarse entre todos (atención) + capas] × muchas veces → porcentajes para el pedacito siguiente → elegir uno con la perilla de la creatividad**, pegarlo y repetir.
> Las fotos entran a la misma cadena cortadas en parches.

**Entregable.** Sube a AULA-USM este notebook **ejecutado** (Partes 1 a 4), con las respuestas y un párrafo de **6 a 8 líneas**:

> *¿Cómo decide una IA qué decir, y por qué puede equivocarse con total seguridad? Usa al menos un ejemplo que hayas visto hoy.*

**Tu párrafo:** (escribe aquí, 6 a 8 líneas)

In [ ]:
print(f"Tiempo total del notebook: {(time.time() - T0) / 60:.1f} min")

---
## Extras opcionales

### E1 — Mirar la atención real (diapos 7–8)

> **En palabras simples (diapos 7 y 8).** Dentro del modelo, cada pedacito **mira a todos los anteriores** para decidir qué significa (eso es la atención). Lo hace con varias «cabezas» a la vez, y cada cabeza se fija en algo distinto.

`mapa_atencion(frase, capa, cabeza)` muestra, para una capa y una cabeza del que solo completa (tiene 24 capas con 14 cabezas cada una), cuánto mira cada pedacito (fila) a cada uno de los anteriores (columna). Más oscuro = mira más. Solo mirar: cambia la capa (0 a 23) y la cabeza (0 a 13) y busca patrones. La capa 8, cabeza 7, casi siempre mira **al pedacito anterior** (la diagonal de al lado).

In [ ]:
mapa_atencion("El animal no cruzó la calle porque estaba muy cansado", capa=8, cabeza=7)

In [ ]:
mapa_atencion("El animal no cruzó la calle porque estaba muy cansado", capa=0, cabeza=0)

### E2 — La atención hecha a mano

Si quieres ver la matemática que hay detrás (Anexos A y B del deck), el laboratorio antiguo construye la atención a mano con NumPy: abre [`_anterior/Lab14_DistilGPT2_estudiante.ipynb`](https://colab.research.google.com/github/diegoalrv/ml-ein143a25/blob/main/14/_anterior/Lab14_DistilGPT2_estudiante.ipynb), Partes A y B.

### E3 — Pensando en tu proyecto

Si tu tópico es **Transformers**, este lab ya trae piezas útiles: clasificar textos o fotos **sin entrenar** y buscar por significado. Pero, como en todo el curso, hay que **medirlo** (Clase 9). Ejemplo: clasificar 10 frases eligiendo, para cada una, la etiqueta con más `parecido`, y calcular la exactitud.

In [ ]:
ejemplos = [
    ("Mi perro mueve la cola cuando llego", "animales"), ("El gato se subió al techo", "animales"),
    ("Viña del Mar se llena en verano", "ciudades de Chile"), ("Temuco queda al sur de Concepción", "ciudades de Chile"),
    ("Hoy almorcé porotos con riendas", "comida"), ("El mote con huesillo es lo mejor del verano", "comida"),
    ("El partido se fue a penales", "fútbol"), ("Le sacaron tarjeta roja al defensa", "fútbol"),
    ("La ballena salió a respirar cerca del bote", "animales"), ("Las sopaipillas pasadas son para la lluvia", "comida"),
]
ETIQUETAS = {"animales": "una frase sobre animales", "ciudades de Chile": "una frase sobre una ciudad de Chile",
             "comida": "una frase sobre comida", "fútbol": "una frase sobre fútbol"}
filas = []
for frase, real in ejemplos:
    predicha = max(ETIQUETAS, key=lambda g: parecido(frase, ETIQUETAS[g]) or 0)
    filas.append((frase, real, predicha, "✔" if predicha == real else "✘"))
tabla = pd.DataFrame(filas, columns=["frase", "real", "predicha", "¿acertó?"])
display(tabla)
print(f"Exactitud: {(tabla['real'] == tabla['predicha']).mean():.0%} (sin entrenar nada)")